# 06 LLM Fine-Tuning for Prescriptive Maintenance (Qwen2.5-7B)

This notebook fine-tunes Qwen2.5-7B-Instruct using Unsloth and LoRA adapters on synthetic maintenance data to generate Indonesian Standard Operating Procedures (SOP).

## 1. Install Dependencies

In [ ]:
# Install Unsloth and fine-tuning dependencies
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Load Base Model & Configure LoRA Adapters

In [ ]:
# from unsloth import FastLanguageModel
# import torch

# max_seq_length = 2048
# dtype = None
# load_in_4bit = True

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = load_in_4bit,
# )

# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 16,
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
#                       "gate_proj", "up_proj", "down_proj"],
#     lora_alpha = 16,
#     lora_dropout = 0,
#     bias = "none",
#     use_gradient_checkpointing = "unsloth",
#     random_state = 3407,
# )

## 3. Load & Format Dataset (Alpaca Prompt)

In [ ]:
# from datasets import load_dataset

# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""

# EOS_TOKEN = tokenizer.eos_token
# def formatting_prompts_func(examples):
#     instructions = examples["instruction"]
#     inputs       = examples["input"]
#     outputs      = examples["output"]
#     texts = []
#     for instruction, input, output in zip(instructions, inputs, outputs):
#         text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
#         texts.append(text)
#     return { "text" : texts, }

# dataset = load_dataset("json", data_files="../src/sop_dataset.jsonl", split="train")
# dataset = dataset.map(formatting_prompts_func, batched = True,)

## 4. Fine-Tune Model with SFTTrainer

In [ ]:
# from trl import SFTTrainer
# from transformers import TrainingArguments
# from unsloth import is_bfloat16_supported

# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2,
#     packing = False,
#     args = TrainingArguments(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 5,
#         max_steps = 25,
#         learning_rate = 2e-4,
#         fp16 = not is_bfloat16_supported(),
#         bf16 = is_bfloat16_supported(),
#         logging_steps = 1,
#         optim = "adamw_8bit",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 3407,
#         output_dir = "outputs",
#     ),
# )

# trainer_stats = trainer.train()

## 5. Save LoRA Adapters & Export Artifacts

In [ ]:
# model.save_pretrained("qwen-sop-model")
# tokenizer.save_pretrained("qwen-sop-model")
# print("LoRA adapter model saved successfully in 'qwen-sop-model'/")